# CDS 303 Final Project 2: Data Integration, Research Question Refinement & PCA
**Team:** DongHyeok Kim, DongJun Lee

In [4]:
import json
import numpy as np
import pandas as pd
from matplotlib.path import Path

econ = pd.read_excel('data/Econ_2024ACS5YR_NTA.xlsx', sheet_name='EconData')
hous = pd.read_excel('data/Hous_2024ACS5YR_NTA.xlsx', sheet_name='HousData')

econ = econ[econ['NTAType'] == 0][['GeoID', 'PBwPvP', 'CvLFUEm2P', 'CW_PbTrnsP', 'MdHHIncE']]
hous = hous[hous['NTAType'] == 0][['GeoID', 'OOcHU1P', 'OcPR1plP', 'GRPI30plP', 'MdGRE']]

nta = econ.merge(hous, on='GeoID')
nta.columns = ['GeoID', 'poverty_pct', 'unemployment_pct', 'transit_commute_pct', 'median_hh_income',
               'owner_occupied_pct', 'crowding_pct', 'rent_burden_pct', 'median_gross_rent']
print(nta.shape, nta['GeoID'].is_unique)

(197, 9) True


In [3]:
import os
print(os.listdir('data'))

['Demographic__Social__Economic__and_Housing_Profiles_by_Community_District_PUMA 3.xlsx', 'Econ_2024ACS5YR_NTA (2).xlsx', 'Hous_2024ACS5YR_NTA.xlsx', 'listings (1).csv']


In [5]:
listings = pd.read_csv('data/listings.csv')
df = listings.dropna(subset=['price']).copy()
print(len(listings), '->', len(df))

29918 -> 19406


In [7]:
# assign each listing to the NTA whose boundary polygon contains its coordinates
shapes = json.load(open('data/nta2020_boundaries.geojson'))['features']
points = df[['longitude', 'latitude']].values
codes = np.full(len(df), None, dtype=object)

for s in shapes:
    g = s['geometry']
    polys = [g['coordinates']] if g['type'] == 'Polygon' else g['coordinates']
    for rings in polys:
        inside = Path(np.array(rings[0])).contains_points(points)
        for hole in rings[1:]:
            inside &= ~Path(np.array(hole)).contains_points(points)
        codes[inside] = s['properties']['NTACode']

df['GeoID'] = codes
print('Listings with no NTA match:', df['GeoID'].isnull().sum())

Listings with no NTA match: 3


In [8]:
# merge neighborhood values onto each listing and check for duplicated rows
merged = df.merge(nta, on='GeoID', how='left')
print(len(df), len(merged), merged['id'].duplicated().sum())

# drop listings with no neighborhood values or no minimum_nights
merged = merged.dropna(subset=list(nta.columns[1:]) + ['minimum_nights'])
print(merged.shape, merged['GeoID'].nunique())

merged.to_csv('data/listings_merged.csv', index=False)

19406 19406 0
(19331, 28) 195
